# 02 - Quality Governance

Evalua reglas de calidad y crea tablas `gov_*` para gobierno y acciones.

In [ ]:
from datetime import datetime, timezone
import json
import uuid

from pyspark.sql.functions import col

threshold = 95.0
asset_id = str(uuid.uuid4())
run_id = str(uuid.uuid4())
now = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
fact = spark.table("curated_fact_transactions")
total_rows = fact.count()

rules = [
    ("Customer master referenced", "Integrity", "Critical", "flag_customer_in_crm = 1", col("flag_customer_in_crm") == "1"),
    ("Product master referenced", "Integrity", "Critical", "flag_product_in_master = 1", col("flag_product_in_master") == "1"),
    ("Movement date parsed", "Validity", "High", "flag_fecha_parseada = 1", col("flag_fecha_parseada") == "1"),
    ("Amount parsed", "Validity", "High", "flag_importe_parseado = 1", col("flag_importe_parseado") == "1"),
    ("Currency is EUR", "Consistency", "Medium", "moneda = EUR", col("moneda") == "EUR"),
    ("Fact id complete", "Completeness", "Critical", "fact_id IS NOT NULL", col("fact_id").isNotNull()),
    ("Product category complete", "Completeness", "Medium", "category IS NOT NULL", col("category").isNotNull() & (col("category") != "")),
    ("Fact id unique", "Uniqueness", "Critical", "fact_id is unique", ~col("fact_id").isin([r["fact_id"] for r in fact.groupBy("fact_id").count().where(col("count") > 1).select("fact_id").collect()])),
]

def remediation(rule_name):
    hints = {
        "Customer master referenced": "Normalize customer identifiers and reconcile unknown customer records with CRM.",
        "Product master referenced": "Update the product master or quarantine orphan product movements.",
        "Movement date parsed": "Standardize source date formats before curated publication.",
        "Amount parsed": "Strip currency symbols and normalize decimal separators.",
    }
    return hints.get(rule_name, "Review source records and update the data quality contract.")

rule_rows = []
result_rows = []
action_rows = []

for rule_name, dimension, severity, expression, predicate in rules:
    passed = fact.where(predicate).count()
    failed = total_rows - passed
    score = round((passed / total_rows * 100) if total_rows else 100, 2)
    rule_id = str(uuid.uuid4())
    result_id = str(uuid.uuid4())
    rule_rows.append((rule_id, asset_id, rule_name, None, dimension, expression, severity, threshold, "data.owner@contoso.com", remediation(rule_name), True, now))
    result_rows.append((result_id, run_id, rule_id, total_rows, passed, failed, 0, score, f"SELECT * FROM curated_fact_transactions WHERE NOT ({expression})", "{}", "Passed" if score >= threshold else "Failed"))
    if score < threshold and severity in ("Critical", "High"):
        action_rows.append((str(uuid.uuid4()), asset_id, result_id, f"Remediate: {rule_name}", f"Quality score {score} is below threshold {threshold}.", "data.steward@contoso.com", None, "Open", "Impacts trusted BI, scoring and data agent answers.", remediation(rule_name), now, None))

global_score = round(sum(row[7] for row in result_rows) / len(result_rows), 2) if result_rows else 100.0
run_rows = [(run_id, asset_id, now, now, "Completed", total_rows, int(total_rows * global_score / 100), int(total_rows * (100 - global_score) / 100), global_score, "fabric_notebook")]

rule_schema = "rule_id string, asset_id string, rule_name string, column_name string, quality_dimension string, rule_expression string, severity string, threshold_pct double, owner_upn string, remediation_hint string, is_active boolean, created_at string"
run_schema = "run_id string, asset_id string, started_at string, finished_at string, status string, total_rows long, passed_rows long, failed_rows long, global_score double, execution_source string"
result_schema = "result_id string, run_id string, rule_id string, evaluated_rows long, passed_rows long, failed_rows long, ignored_rows long, score double, sample_query string, sample_json string, status string"
action_schema = "action_id string, asset_id string, result_id string, title string, description string, owner_upn string, due_date string, status string, business_impact string, remediation string, created_at string, closed_at string"

spark.createDataFrame(rule_rows, rule_schema).write.mode("overwrite").format("delta").saveAsTable("gov_quality_rule")
spark.createDataFrame(run_rows, run_schema).write.mode("overwrite").format("delta").saveAsTable("gov_quality_run")
spark.createDataFrame(result_rows, result_schema).write.mode("overwrite").format("delta").saveAsTable("gov_quality_result")
spark.createDataFrame(action_rows, action_schema).write.mode("overwrite").format("delta").saveAsTable("gov_action")

summary = {
    "asset_id": asset_id,
    "global_score": global_score,
    "failed_rules": sum(1 for row in result_rows if row[10] == "Failed"),
    "open_actions": len(action_rows),
}
mssparkutils.fs.mkdirs("Files/generated")
mssparkutils.fs.put("Files/generated/quality_summary.json", json.dumps(summary, indent=2), True)

display(spark.table("gov_quality_result").orderBy("score"))